In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [8]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

label_encoder = LabelEncoder()
for col in train_df.select_dtypes(include=['object']).columns:
    train_df[col] = train_df[col].astype(str)
    train_df[col] = label_encoder.fit_transform(train_df[col])

print("Label encoded successfully! Shape:", train_df.shape)

Label encoded successfully! Shape: (1460, 81)


In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')

nominal_cols = train_df.select_dtypes(include=['object']).columns.tolist()
if nominal_cols:
    train_df = pd.get_dummies(train_df, columns=nominal_cols, drop_first=True)

print("One-Hot Encoded successfully! Shape:", train_df.shape)

One-Hot Encoded successfully! Shape: (1460, 246)


In [10]:
train_df = pd.read_csv('train.csv')

target_col = 'SalePrice'

if target_col in train_df.columns:
    train_df.fillna(0, inplace=True)

    X = train_df.drop(columns=[target_col])
    y = train_df[target_col]

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    
    X_train_numeric = X_train.select_dtypes(include=[np.number])
    X_val_numeric = X_val.select_dtypes(include=[np.number])

    X_train_scaled = scaler.fit_transform(X_train_numeric)
    X_val_scaled = scaler.transform(X_val_numeric)

    from sklearn.linear_model import LinearRegression
    from sklearn.metrics import mean_squared_error, r2_score

    model = LinearRegression()
    model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_val_scaled)
    print("MSE:", mean_squared_error(y_val, y_pred))
    print("R2 Score:", r2_score(y_val, y_pred))
else:
    print(f"Error: '{target_col}' not found in DataFrame columns. Available columns are: {list(train_df.columns)}")

MSE: 1303361738.349666
R2 Score: 0.8300774043836088


In [11]:
## Fixing Target Column and Running Logistic Regression

In [ ]:
# Replace 'SalePrice' with your actual target column name if needed
target_col = 'SalePrice'

# If your target is continuous (like SalePrice), we can bin it into categories for Logistic Regression, 
# or choose a categorical column like 'MSZoning' or 'BldgType' as your target.
# For demonstration, let's look at the first few column names to find a good categorical target if needed:
print("Available columns:", train_df.columns[:10])

# Fill missing values
train_df.fillna(0, inplace=True)

# If 'SalePrice' is numeric, let's convert it to binary classes (e.g., 1 if above median, else 0) for Logistic Regression:
if train_df[target_col].dtype in ['int64', 'float64']:
    median_val = train_df[target_col].median()
    train_df[target_col] = (train_df[target_col] > median_val).astype(int)

X = train_df.drop(columns=[target_col])
y = train_df[target_col]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

y_pred = log_reg.predict(X_val_scaled)
print("Accuracy:", accuracy_score(y_val, y_pred))
print("\nClassification Report:\n", classification_report(y_val, y_pred))

Available columns: Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1'],
      dtype='object')
Accuracy: 0.934931506849315

Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.91      0.94       161
           1       0.89      0.97      0.93       131

    accuracy                           0.93       292
   macro avg       0.93      0.94      0.93       292
weighted avg       0.94      0.93      0.94       292

